# start

In [1]:
import os
import pandas as pd
import polars as pl
from astropy.io import fits
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# reads the fits headers for a file and returns 4 lists of the headre number, tags, values and descriptions
def get_fits_headers(full_file_name, verbose = False):

    try:
        list_hdu = [] # header number
        list_tag = [] # element tag
        list_value = [] # value
        list_description = [] # long form description

        if verbose:
            print(f"Reading: {full_file_name}")

        with fits.open(full_file_name) as hdul:
            for i, hdu in enumerate(hdul):
                x = repr(hdu.header).split('\n')
                for j in x:
                    list_hdu.append(i)
                    j = j.replace('=',',')
                    j = j.replace('/',',')
                    y = j.split(',')
                    y = [item.strip() for item in y if item.strip()]
                    if len(y) == 2:
                        y.append('')

                    list_tag.append(y[0])
                    list_value.append(y[1])
                    if y[2] == '[km':
                        if y[0] == 'WINDSPEE':
                            y[2] = 'km wind speed'
                    list_description.append(y[2])
        
        return(list_hdu, list_tag, list_value, list_description)
    
    except Exception as e:
        print(f"Error: {e}")
        return None

In [3]:
# for a list of values, returns a list of normalised values between 0 and 1, and the min and max values
def get_normalised(list_of_values, verbose = False) :
    try:
        if not list_of_values:
            print("List of values is empty.")
            return [], None, None
        
        if verbose:
            print(f"Original values: {list_of_values}")
            
        min_value = min(list_of_values)
        max_value = max(list_of_values)
        if verbose:
            print(f"Min: {min_value}, Max: {max_value}")
        
        normal_values = [(value - min_value) / (max_value - min_value) for value in list_of_values]
        if verbose:
            print(f"Normalised values: {normal_values}")    
            
        return normal_values, min_value, max_value
        # return [(value - min_value) / (max_value - min_value) for value in list_of_values]
    except Exception as e:
        print(f"Error normalizing values: {e}")
        return [], None, None

In [4]:
# get the header data from the fits files as a df
def get_fits_headers_data(star_info, verbose = False):

    star = star_info[0]
    obs_date = star_info[1]
    folder = star_info[2]
    exclude_files = star_info[3]

    # create a dataframe of the fits headers for the star and date
    file_path = os.path.join(os.getcwd(), star, folder)
    fits_files = [f for f in os.listdir(file_path) if f.endswith(".fits.fz")]

    try:
        df = None
        
        for fileno, file_name in enumerate(fits_files):
            if fileno in exclude_files:
                continue
            
            # Create a DataFrame from the lists
            list_hdu, list_tag, list_value, list_description = get_fits_headers(os.path.join(file_path, file_name))
            if df is None:
                # assuming this will always be the same for all files
                frame_index = list_tag.index('FRAMENUM')

            # get the frame number as a column header
            frame_label = f"Frame-{list_value[frame_index]}"

            if df is None:
                # First file: build full DataFrame
                df = pl.DataFrame({
                    'HDU': list_hdu,
                    'Description': list_description,
                    'Tag': list_tag,
                    frame_label: list_value
                })
            else:
                # Append new column to existing DataFrame
                df = df.with_columns(
                    pl.Series(frame_label, list_value)
                ) 
                           
        return(df)
    except Exception as e:
            print(f"Error reading {fileno} FITS file: {e}")

In [5]:
def get_plot_data(df_polars, plot_rows, verbose = False):

    try:
        if len(plot_rows) == 0:
            # values to plot
            plot_rows = [#'Effective mean airmass', 
                        #'[mbar] Atmospheric pressure',
                        #'[%] Current percentage humidity',
                        #'[deg C] External temperature',
                        #'[deg] Enclosure azimuth',
                        #'km wind speed',
                        '[arcsec] Frame FWHM in arcsec',
                        #'[arcsec] Autoguider FWHM',
                        'Mean image ellipticity (1-B',
                        #'[deg] PA of mean image ellipticity',
                    ]

        plot_columns = [col for col in df_polars.columns if col.startswith("Frame-")]

        plot_xtext = (
            df_polars
            .filter(pl.col("Description") == '[UTC] Start date and time of the observati')
            .select([
                pl.col(col).cast(pl.Utf8).str.slice(12, 8)  # 10 = length from 12 to 20
                for col in plot_columns
            ])
            .row(0) 
        )

        plot_xtext = [f"{col.removeprefix('Frame-')} : {time}" for col, time in zip(plot_columns, plot_xtext)]

        plot_data = (
            df_polars
            .filter(pl.col("Description").is_in(plot_rows))
            .select(['Description'] + plot_columns)
        )

        df_plot = plot_data.to_pandas()

        # return a pandas dataframe with the plot data
        return df_plot, plot_columns, plot_xtext
    
    except Exception as e:
        print(f"Error getting plot data: {e}")
        return None, None

In [6]:
def get_chart(star_info, df_plot, plot_columns, plot_xtext,  verbose = True):

    fig = go.Figure()
    color_list = ['blue', 'green', 'brown', 'red', 'red', 'brown']

    try:
        # number of points on the x axis
        x = list(range(len(plot_columns)))

        for i in range(len(df_plot)):
            yvalue = df_plot.iloc[i]['Description']
            print(f"Processing: {yvalue}")

            list_vals = df_plot.iloc[i][plot_columns].values.astype(float).tolist()
            y, miny, maxy = get_normalised(list_vals, verbose = False)
            yname = f'{yvalue} {miny:.2f} - {maxy:.2f}'

            if yvalue == 'km wind speed':
                fig.add_trace(go.Scatter(x=x, y=y, name=yname,
                                        fill='tozeroy',  # Fill area down to y=0
                                        mode='none',      # No line or markers
                                        fillcolor='rgba(0, 100, 250, 0.4)',
                                        text=plot_xtext,           # This sets hover text per point
                                        hoverinfo='text+y'))  # Semi-transparent blue
            else:
                fig.add_trace(go.Scatter(x=x, y=y, name=yname,
                                        line=dict(color=color_list[i]),
                                        text=plot_xtext,           # This sets hover text per point
                                        hoverinfo='text+y'))     # Show only the custom text and y value))

        # Set the tick labels using xtexts
        fig.update_layout(
            xaxis=dict(
                tickmode='array',
                tickvals=x[::10],       # Every 10th x value
                ticktext=plot_xtext[::10],  # Matching every 10th label
                tickangle=45            # Rotate labels 45 degrees
            ),
            title=f'BANZAI Headers measures for {star_info[0]}, {star_info[1]}',
        )
        fig.show()
    except Exception as e:
        print(f"Error creating plot: {e}")

# get BANZAI header information as CSV

## note the interpretation of 
[arcsec] Frame FWHM in arcsec header - L1FWHM
“How wide a typical star appears on this image, in terms of angular size on the sky.”
Affected by optics, seeing, focus, tracking

In [8]:
# which dataset to use

star_info_wasp = ['WASP-123b','2024-07-01', "lco_data-20250414-660",[239]]
star_info_hats = ['HATS-38b','2025-03-26',"lco_data-20250407-211",[]]

star_info = star_info_wasp

df = get_fits_headers_data(star_info, verbose = True)
df_plot, plot_columns, plot_xtext = get_plot_data(df, plot_rows = [], verbose = True)
get_chart(star_info, df_plot, plot_columns, plot_xtext)

Processing: [arcsec] Frame FWHM in arcsec
Processing: Mean image ellipticity (1-B
